# Dataset Audit

## Imports

In [21]:
import sys
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd

In [22]:
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [23]:
from src.utils.config.config import Config

## Config

In [24]:
config_loader = Config()
cfg = config_loader.load_config()

dataset_root = cfg.project_root / cfg.paths.dataset / "CafeV1"
clips_root   = dataset_root / "Clips"

with open(dataset_root / "action_classes.json", "r") as f:
    ACTION_NAME_TO_ID = json.load(f)

ACTION_ID_TO_NAME = {v: k for k, v in ACTION_NAME_TO_ID.items()}

OBJECT_CATEGORY_NAMES = {
    1: "person",
    2: "laptop",
    3: "cell phone",
    4: "book",
}

## Helpers

In [ ]:
def list_viewpoints():
    return sorted([p.name for p in clips_root.iterdir() if p.is_dir()], key=int)

def list_clips(viewpoint):
    vp_path = clips_root / viewpoint
    return sorted([p.name for p in vp_path.iterdir() if p.is_dir()], key=int)

def count_objects(anns_path):
    with open(anns_path, "r") as f:
        coco = json.load(f)
    counts = defaultdict(int)
    for ann in coco.get("annotations", []):
        counts[ann["category_id"]] += 1
    return counts

def count_actions(hoi_path):
    with open(hoi_path, "r") as f:
        data = json.load(f)
    counts = defaultdict(int)
    for seg in data.get("annotations", []):
        counts[seg["action_id"]] += 1
    return counts

def audit():
    object_counts = defaultdict(lambda: defaultdict(int))   # vp -> cat_id -> n
    action_counts = defaultdict(lambda: defaultdict(int))   # vp -> action_id -> n
    clips_per_vp  = defaultdict(int)
    missing_anns  = []
    missing_hoi   = []

    for vp in list_viewpoints():
        for clip in list_clips(vp):
            clips_per_vp[vp] += 1
            cdir = clips_root / vp / clip
            anns = cdir / "anns.json"
            hoi  = cdir / "hoi-anns.json"

            if anns.exists():
                for cat_id, n in count_objects(anns).items():
                    object_counts[vp][cat_id] += n
            else:
                missing_anns.append(f"vp{vp}/clip{clip}")

            if hoi.exists():
                for action_id, n in count_actions(hoi).items():
                    action_counts[vp][action_id] += n
            else:
                missing_hoi.append(f"vp{vp}/clip{clip}")

    return object_counts, action_counts, clips_per_vp, missing_anns, missing_hoi

def to_frame(counts_by_vp, classes):
    viewpoints = sorted(counts_by_vp.keys(), key=int)
    df = pd.DataFrame(
        {vp: [counts_by_vp[vp].get(cid, 0) for cid in classes] for vp in viewpoints},
        index=[classes[cid] for cid in classes],
    )
    df["TOTAL"] = df.sum(axis=1)
    df.loc["TOTAL"] = df.sum(axis=0)
    return df

## Run audit

In [26]:
object_counts, action_counts, clips_per_vp, missing_anns, missing_hoi = audit()

n_vp = len(clips_per_vp)
n_clips = sum(clips_per_vp.values())
print(f"Found {n_vp} viewpoints, {n_clips} clips total.")
print("Clips per viewpoint:", dict(sorted(clips_per_vp.items(), key=lambda kv: int(kv[0]))))

Found 14 viewpoints, 126 clips total.
Clips per viewpoint: {'1': 9, '2': 9, '3': 9, '4': 9, '5': 9, '6': 9, '7': 9, '8': 9, '9': 9, '10': 9, '11': 9, '12': 9, '13': 9, '14': 9}


## Object instances per viewpoint

In [27]:
to_frame(object_counts, OBJECT_CATEGORY_NAMES)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,TOTAL
person,3279,3404,2720,2419,3237,3212,3323,3075,2446,3421,3236,2479,3432,3412,43095
laptop,575,716,159,224,391,100,481,592,0,280,0,0,272,437,4227
cell phone,1549,1203,2049,1361,1750,1223,1817,2074,1401,1649,1690,1080,2085,1964,22895
book,1619,691,1698,1431,1598,932,1793,1555,593,633,803,68,1449,1178,16041
TOTAL,7022,6014,6626,5435,6976,5467,7414,7296,4440,5983,5729,3627,7238,6991,86258


## Action segments per viewpoint

This is the matrix that determines whether viewpoint-disjoint k-fold is safe.
Every column should ideally contain non-zero counts for every action class.

In [28]:
to_frame(action_counts, ACTION_ID_TO_NAME)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,TOTAL
idle,69,106,49,54,59,76,58,44,73,90,86,112,86,92,1054
using_laptop,2,4,2,2,3,4,4,4,0,0,0,0,5,2,32
using_phone,32,30,33,25,29,27,37,40,56,48,49,55,54,45,560
reading,26,20,33,27,32,20,30,36,5,6,9,7,17,9,277
TOTAL,129,160,117,108,123,127,129,124,134,144,144,174,162,148,1923


## Balance check

In [29]:
skewed = []
for vp in sorted(action_counts.keys(), key=int):
    missing = [ACTION_ID_TO_NAME[aid] for aid in ACTION_ID_TO_NAME
               if action_counts[vp].get(aid, 0) == 0]
    if missing:
        skewed.append((vp, missing))

if not skewed:
    print("All viewpoints contain all action classes — safe for viewpoint-disjoint k-fold.")
else:
    print(f"{len(skewed)} viewpoint(s) are missing one or more action classes:")
    for vp, missing in skewed:
        print(f"  vp{vp}: missing {missing}")
    print("\nConsider grouping these with complementary viewpoints when forming folds,")
    print("or excluding them and documenting the choice in your methods section.")

4 viewpoint(s) are missing one or more action classes:
  vp9: missing ['using_laptop']
  vp10: missing ['using_laptop']
  vp11: missing ['using_laptop']
  vp12: missing ['using_laptop']

Consider grouping these with complementary viewpoints when forming folds,
or excluding them and documenting the choice in your methods section.


## Missing annotation files

In [30]:
if missing_anns:
    print(f"{len(missing_anns)} clips missing anns.json:")
    for p in missing_anns:
        print(f"  {p}")
else:
    print("All clips have anns.json.")

print()

if missing_hoi:
    print(f"{len(missing_hoi)} clips missing hoi-anns.json (not yet HOI-annotated):")
    for p in missing_hoi:
        print(f"  {p}")
else:
    print("All clips have hoi-anns.json.")

All clips have anns.json.

All clips have hoi-anns.json.
